In [2]:
import os
import re
from typing import Any, Dict

import mlflow
from rich import print as rptint
from dotenv import load_dotenv
import numpy as np
from pymongo import MongoClient

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers.hybrid_search import MongoDBAtlasHybridSearchRetriever

mlflow.langchain.autolog()
load_dotenv()

MONGODB_URI = os.getenv('MONGODB_URI')
DB_NAME = 'agent_db'
COLLECTION_NAME = 'cxi-travel-ins-collection'
ATLAS_VECTOR_SEARCH_INDEX_NAME = "cxi-travel-ins-qa-vector-index"

client = MongoClient(MONGODB_URI)
MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

In [3]:
client = MongoClient(MONGODB_URI)
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)
vector_store = MongoDBAtlasVectorSearch(
    collection=MONGODB_COLLECTION,
    embedding=embeddings,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    relevance_score_fn="cosine",
    embedding_key="embedding",
    text_key="text"
)

In [6]:
def build_pre_filter(where: Dict[str, Dict[str, Any]] | None = None) -> Dict[str, Any]:
    ALLOWED_FIELDS = {"article", "anchor_page"}
    ALLOWED_OPS = {"$eq", "$ne", "$in", "$nin", "$gt", "$gte", "$lt", "$lte"}
    
    if not where:
        return {}
    clauses = []
    for field, ops in where.items():
        if field not in ALLOWED_FIELDS:
            raise ValueError(f"不支援欄位: {field}")
        if not isinstance(ops, dict):
            raise ValueError(f"{field} 的條件必須是 dict")
        field_ops = {}
        for op, value in ops.items():
            if op not in ALLOWED_OPS:
                raise ValueError(f"{field} 不支援運算子: {op}")
            field_ops[op] = value
        clauses.append({field: field_ops})
    if len(clauses) == 1:
        return clauses[0]
    return {"$and": clauses}

In [17]:
@tool(response_format="content_and_artifact")
def retreve_context(query: str, where: Dict[str, Dict[str, Any]] | None = None) -> str:
    """
        依照使用者問題，從旅平險條款知識庫檢索相關內容。
        何時使用:
        - 使用者詢問保險條款、理賠條件、理賠流程、保障範圍、除外責任等問題時。
        - 需要先查文件再回答時。
        參數:
        - query (str): 必填。自然語言查詢，例如「行李遺失後如何申請理賠」。
        - where (dict | None): 選填。Mongo 風格篩選條件。
        支援欄位:
            - article
            - anchor_page
        支援運算子:
            - $eq, $ne, $in, $nin, $gt, $gte, $lt, $lte
        where 格式:
        {
        "<field>": {"<operator>": <value>},
        ...
        }
        範例:
        1) 指定條文
        {"article": {"$eq": "第二十七條"}}
        2) 篩頁碼區間
        {"anchor_page": {"$gte": 10, "$lte": 20}}
        3) 同時篩條文與頁碼
        {
            "article": {"$in": ["第二十七條", "第二十八條"]},
            "anchor_page": {"$gte": 5}
        }
        回傳:
        - str: 由多段條款內容組成的純文字（以空行分隔）。
        注意:
        - 不支援未列出的欄位或運算子，否則會拋出 ValueError。
        """
    pre_filter = build_pre_filter(where)
    retriever = MongoDBAtlasHybridSearchRetriever(
        vectorstore=vector_store,
        search_index_name="search_index",
        top_k=5,
        fulltext_penalty=50,
        vector_penalty=50,
        pre_filter=pre_filter,
        post_filter=[
            {
                '$project': {
                    'embedding': 0,
                }
            }
        ]
    )
    retrieved_docs = retriever.invoke(query)
    serialized = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    return serialized, retrieved_docs

In [20]:
tools = [retreve_context]
prompt = (
    "You are a helpful assistant that can answer questions about the insurance policy."
    "Use the tool to retrieve the context from the MongoDB Atlas Vector Search."
    "If you don't know the answer, just say 'I don't know'."
    "使用繁體中文回答"
)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt,
    
)

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "行李遺失後應該如何申請理賠？"}]})